[Reference](https://medium.com/@swapnil29071999/memory-knowledge-grounding-your-ai-agent-in-reality-51bc83fb2f23$0)

# 1.1. Update the Orchestrator’s Code

``` requirements.txt
fastapi
uvicorn
langchain
langchain-google-vertexai
google-cloud-firestore # <-- Add this line
```

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
from langchain_google_vertexai import ChatVertexAI
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.memory import ChatMessageHistory
from langchain_community.chat_message_histories import FirestoreChatMessageHistory

from tools import get_order_details, get_current_time_tool # Your tools from Part 3
# --- 1. Agent Setup (mostly unchanged) ---
llm = ChatVertexAI(model_name="gemini-1.5-pro-preview-0409", temperature=0)
tools = [get_order_details, get_current_time_tool]
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. You are polite and friendly."),
    # The 'chat_history' placeholder will be filled by the agent executor
    ("placeholder", "{chat_history}"),
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])
agent = create_tool_calling_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True)

# --- 2. API Server Setup (now with session_id) ---
app = FastAPI(title="Agent Orchestrator", version="1.0.0")
class AgentRequest(BaseModel):
    input: str
    session_id: str # <-- Add session_id to identify the conversation
# --- 3. The Core Logic Update ---
@app.post("/invoke", summary="Invoke the AI Agent")
def invoke_agent(request: AgentRequest):
    """
    Receives a user query and a session_id, then runs it through the agent.
    The session_id is used to retrieve and update the conversation history.
    """
    # Get the conversation history from Firestore for the given session_id
    chat_history = FirestoreChatMessageHistory(
        session_id=request.session_id,
        collection="agent-chat-history", # Name of the Firestore collection
        # Project ID will be inferred from the environment
    )
    # Invoke the agent, now with memory
    response = agent_executor.invoke({
        "input": request.input,
        "chat_history": chat_history.messages,
    })
    # The LangChain agent executor with Firestore history automatically
    # saves the latest human and AI message to the database.
    # No manual saving is needed here!
    return {"output": response["output"]}

```
# Replace 'your-gcp-project-id' with your actual Project ID
PROJECT_ID="your-gcp-project-id"
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:my-agent-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
    --role="roles/datastore.user"
```

```

# First call
curl -X POST "${SERVICE_URL}/invoke" \
-H "Content-Type: application/json" \
-d '{ "input": "My name is Bob.", "session_id": "user123" }'


# Second call, with the same session_id
curl -X POST "${SERVICE_URL}/invoke" \
-H "Content-Type: application/json" \
-d '{ "input": "What is my name?", "session_id": "user123" }'
```

# Part 2: Grounding in Reality with Vertex AI Search (RAG)


``` requirements.txt
fastapi
uvicorn
langchain
langchain-google-vertexai
google-cloud-firestore
google-cloud-discoveryengine # <-- Add this line
```

In [2]:
# ... (keep the other tools from before)
from langchain_google_vertexai import VertexAISearchRetriever
from langchain.tools import create_retriever_tool
# 1. Initialize the retriever
#    Replace 'your-data-store-id' and 'your-project-id'
retriever = VertexAISearchRetriever(
    project_id="your-gcp-project-id",
    location="global",
    data_store_id="your-data-store-id",
    engine_data_type=1 # 1 for unstructured data
)
# 2. Create a tool from the retriever
#    The description is VERY important. It tells the agent WHEN to use this tool.
knowledge_base_tool = create_retriever_tool(
    retriever,
    "search_company_knowledge_base",
    "Use this tool to get information about Google Cloud products and services. It is the best source of information for technical questions about Google Cloud."
)

```
gcloud projects add-iam-policy-binding ${PROJECT_ID} \
    --member="serviceAccount:my-agent-sa@${PROJECT_ID}.iam.gserviceaccount.com" \
    --role="roles/discoveryengine.viewer"
```

```
curl -X POST "${SERVICE_URL}/invoke" \
-H "Content-Type: application/json" \
-d '{ "input": "How does Cloud Run scale?", "session_id": "user456" }'
```